# 🚀 Lab 1.3 — Databricks Workspace Setup from Scratch

## Learning Objectives
In this notebook, you will learn:
1. **Package Setup** — Install `mlflow[databricks]>=3.1` and `databricks-openai` in a Databricks notebook
2. **Unity Catalog** — Create a catalog, schema, and volume to hold every tutorial asset
3. **MLflow Experiment** — Register a workspace experiment so traces and runs land in a known location
4. **Foundation Model API** — Verify access to `databricks-claude-sonnet-4` via the OpenAI-compatible client
5. **Sanity-Check Trace** — Use `mlflow.openai.autolog()` to confirm tracing is wired end-to-end

## Prerequisites
- Databricks workspace with Foundation Model APIs enabled
- DBR 15.4 ML LTS (or newer) cluster with internet egress
- Permission to create Unity Catalog catalogs/schemas (or an existing catalog you can use)


---
## Step 1 — Install Packages

We pin `mlflow[databricks]` to `>=3.1` because GenAI evaluation, tracing, and the new judge framework all require MLflow 3.x. `databricks-openai` gives us a drop-in OpenAI-compatible client that authenticates against your workspace automatically.


In [ ]:
# ============================================================================
# 📦 STEP 1 - INSTALL PACKAGES
# ============================================================================

%pip install --quiet "mlflow[databricks]>=3.1" databricks-openai

dbutils.library.restartPython()


---
## Step 2 — Configure Unity Catalog

We create a dedicated catalog and schema so every artifact produced in later labs (eval datasets, judge outputs, traces tables) lives in a predictable location. Replace the defaults below if your workspace requires specific naming.


In [ ]:
# ============================================================================
# 🗂️ SET TUTORIAL NAMESPACE
# ============================================================================

CATALOG  = "genai_eval_tutorial"
SCHEMA   = "module_01"
VOLUME   = "assets"

print(f"Target namespace: {CATALOG}.{SCHEMA}")


In [ ]:
# ============================================================================
# 🗂️ CREATE CATALOG, SCHEMA, AND A MANAGED VOLUME
# ============================================================================

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA  IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME  IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA  {SCHEMA}")

display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))


---
## Step 3 — Set the MLflow Experiment

Every trace, evaluation run, and judge call needs an **experiment** to land in. We use a workspace path under your user folder so it shows up in the MLflow UI sidebar.


In [ ]:
# ============================================================================
# 🧪 STEP 3 - SET THE MLFLOW EXPERIMENT
# ============================================================================

import mlflow

# Resolve the current user so the experiment path is unique per learner
USER_EMAIL = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook()
    .getContext()
    .userName()
    .get()
)

EXPERIMENT_PATH = f"/Users/{USER_EMAIL}/genai-eval-tutorial"
mlflow.set_experiment(EXPERIMENT_PATH)

print(f"MLflow experiment set to: {EXPERIMENT_PATH}")


---
## Step 4 — Verify Foundation Model API Access

`DatabricksOpenAI` is an OpenAI-compatible client that uses your workspace credentials automatically — no API key to manage. We use it to call `databricks-claude-sonnet-4`, a Foundation Model API endpoint that ships with every Databricks workspace.

If this call fails, check:
- Foundation Model APIs are enabled in your workspace
- The cluster has network egress
- You have CAN_QUERY entitlement on the serving endpoint


In [ ]:
# ============================================================================
# 🤖 STEP 4 - VERIFY FOUNDATION MODEL API ACCESS
# ============================================================================

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
client = w.serving_endpoints.get_open_ai_client()

resp = client.chat.completions.create(
    model="databricks-claude-sonnet-4",
    messages=[
        {"role": "user", "content": "What is Delta Lake in one sentence?"}
    ],
)

print(resp.choices[0].message.content)


---
## Step 5 — Sanity-Check Trace with `mlflow.openai.autolog()`

Autologging hooks every OpenAI-compatible call and emits an **MLflow Trace** — the unit of observability you'll inspect, score, and evaluate in every later module. Run the cell, then open the MLflow experiment to confirm the trace appears under the **Traces** tab.


In [ ]:
# ============================================================================
# 🔭 STEP 5 - SANITY-CHECK TRACE WITH `MLFLOW.OPENAI.AUTOLOG()`
# ============================================================================

import mlflow

mlflow.openai.autolog()

resp = client.chat.completions.create(
    model="databricks-claude-sonnet-4",
    messages=[
        {"role": "system", "content": "You are a concise Databricks expert."},
        {"role": "user",   "content": "Name three benefits of MLflow Tracing for GenAI apps."},
    ],
)

print(resp.choices[0].message.content)


### Verify the trace in the UI

1. Click the **Experiments** icon in the left nav
2. Open the experiment at the path printed in Step 3
3. Switch to the **Traces** tab — you should see one trace with the model name, latency, and token counts

You can also fetch the most recent trace programmatically:


In [ ]:
# ============================================================================
# 🔭 VERIFY THE TRACE IN THE UI
# ============================================================================

traces = mlflow.search_traces(experiment_names=[EXPERIMENT_PATH], max_results=1)
display(traces)


---
## Lab Complete

| Check | Status |
| --- | --- |
| `mlflow[databricks]>=3.1` installed | ✅ |
| UC catalog + schema + volume created | ✅ |
| MLflow experiment registered under your user folder | ✅ |
| Foundation Model API call succeeded | ✅ |
| Sanity-check trace visible in the MLflow UI | ✅ |

Proceed to **Module 2** — `Tracing Fundamentals` — once every row is green.


---
## 📝 Summary

In this notebook, we covered:

### 1. Environment Bootstrap
- **`mlflow[databricks]>=3.1`** is the floor for GenAI evaluation — older versions lack `mlflow.genai`.
- **`dbutils.library.restartPython()`** is required after `%pip install` so new packages load into the running kernel.

### 2. Unity Catalog Layout
- **Catalog → Schema → Volume** is the canonical place for tutorial assets — it works for tables, models, and unstructured files.
- Naming the namespace per tutorial (`genai_eval_tutorial.module_01`) keeps later cleanup trivial.

### 3. Experiment Path
- Pinning the experiment under `/Users/<you>/...` makes it visible in the MLflow UI sidebar and isolates runs per learner.

### 4. Tracing Verification
- **`mlflow.openai.autolog()`** is the single line that turns OpenAI-compatible calls into MLflow traces.
- If `search_traces` returns rows, the tracing pipeline is live — every later module depends on this.

### Next Steps
- **Module 2 — Building Evaluation Datasets** — three ways to create the data your judges will score against.
